# Qwen3-4B medical fine-tune, run 3

One session, no laptop needed. It trains a reasoning fine-tune of Qwen3-4B on
about 11,000 examples from eight medical sources, then scores it against the
base model on four benchmarks it never saw -- MedQA, MedMCQA, PubMedQA and
MMLU-medical -- by letter choice and by reasoning.

**Sidebar: Accelerator `GPU T4 x2`, Internet `On`.** Attach `medical-ft-code`
and `medical-ft-data`.

Training stops by itself in time for evaluation. Evaluation stops 40 minutes
before Kaggle's 12-hour limit and reports whatever both models have scored.

In [ ]:
# --- 1. Hardware check, and the session clock ------------------------------
import subprocess, time
from pathlib import Path

# Kaggle ends a GPU session at 12 hours. Every later step measures itself
# against this clock; evaluation stops scoring 40 minutes before the end.
SESSION_START = time.time()
DEADLINE = SESSION_START + 12 * 3600 - 40 * 60

import torch

gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                       "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip().splitlines()
N_GPUS = torch.cuda.device_count()
major, minor = torch.cuda.get_device_capability(0)
for i, gpu in enumerate(gpus):
    print(f"GPU {i}      : {gpu}")
print(f"capability : {major}.{minor}")
print(f"torch      : {torch.__version__}")

if major < 7:
    raise SystemExit(
        f"\nSTOP. Compute capability {major}.{minor} has no kernels in modern "
        "PyTorch builds.\nFIX: kernel-metadata.json pins a T4; if a P100 still "
        "arrived, set sidebar -> Accelerator -> 'GPU T4 x2' and Run All again.")
plan = ("the fine-tune on GPU 0 and the base model on GPU 1, in parallel"
        if N_GPUS > 1 else "both models on GPU 0, one after the other")
print(f"\n{N_GPUS} GPU(s). Training uses GPU 0; evaluation runs {plan}.")

In [ ]:
%%capture
# The exact stack that trained run 1 on this image: Unsloth 2026.9.7, with the
# unsloth_zoo current that day, printed "Transformers: 5.5.0" there, alongside
# trl 0.24.0 and peft 0.19.1. Run 1 got it by installing whatever was newest;
# pinning it is how run 3 gets the same thing again.
!pip install -q "unsloth==2026.9.7" "unsloth_zoo==2026.9.6"
!pip install -q --no-deps "transformers==5.5.0" "trl==0.24.0" "peft==0.19.1"

In [ ]:
# --- 2. Verify the install before spending GPU time on it ------------------
import importlib.metadata as md
import sys

WANT = {"unsloth": "2026.9.7", "transformers": "5.5.0", "trl": "0.24.0",
        "peft": "0.19.1"}
got = {name: md.version(name) for name in WANT}
print("ok |", " | ".join(f"{k} {v}" for k, v in got.items()),
      f"| torch {md.version('torch')}")
wrong = {k: v for k, v in got.items() if v != WANT[k]}
if wrong:
    raise SystemExit(
        f"\nSTOP. Installed {wrong}, expected {WANT}.\nFIX: the install cell did "
        "not take effect. Run > Restart session, then Run All again.")

# peft checks every optional quantization package on the image while it wraps
# each layer, and some checks raise on an old version instead of skipping it:
# run 1's first evaluation died on this image's torchao 0.10.0. Wrap one tiny
# layer in a fresh process, the way training and evaluation will.
LORA_CHECK = """
import torch.nn as nn
from peft import LoraConfig, get_peft_model
class OneLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj = nn.Linear(8, 8)
    def forward(self, x):
        return self.q_proj(x)
get_peft_model(OneLayer(), LoraConfig(r=2, target_modules=["q_proj"]))
"""
check = subprocess.run([sys.executable, "-c", LORA_CHECK], capture_output=True, text=True)
if check.returncode != 0 and "torchao" in check.stderr:
    print("peft rejects this image's torchao; removing it (nothing here uses it)")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
    check = subprocess.run([sys.executable, "-c", LORA_CHECK], capture_output=True, text=True)
if check.returncode != 0:
    raise SystemExit(f"\nSTOP. peft cannot wrap a layer on this image:\n{check.stderr[-2000:]}")
print("ok | peft can wrap a layer on this image")

# transformers 5 replaced group_by_length with train_sampling_strategy. Check
# the class training will really use: Unsloth's patched SFTConfig.
FIELD_CHECK = """
import unsloth, dataclasses
from trl import SFTConfig
print("train_sampling_strategy" in {f.name for f in dataclasses.fields(SFTConfig)})
"""
check = subprocess.run([sys.executable, "-c", FIELD_CHECK], capture_output=True, text=True)
if check.stdout.strip().splitlines()[-1:] != ["True"]:
    raise SystemExit("\nSTOP. SFTConfig has no train_sampling_strategy here:\n"
                     f"{check.stdout[-1000:]}{check.stderr[-1500:]}")
print("ok | SFTConfig accepts train_sampling_strategy")

In [ ]:
# --- 3. Get the code and the data from the attached datasets --------------
import json, os, shlex, shutil, subprocess, sys
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK  = Path("/kaggle/working/ft")
PKG   = WORK / "training"
DATA  = WORK / "data" / "v2"
PKG.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

REQUIRED = {"records", "prompts", "sources", "prepare_data",
            "check_lengths", "budget", "train"}

EVAL_REQUIRED = {"records", "prompts", "sources", "evalcore", "evaluate",
                 "modeling"}

RUN2_REQUIRED = {"records", "prompts", "sources", "sources_v2", "format_v2",
                 "evalcore", "evaluate", "modeling", "train", "budget",
                 "eval_worker", "eval_report", "kaggle_paths"}

DATA_FILES = ("train.jsonl", "eval_medqa.jsonl", "eval_medmcqa.jsonl",
              "eval_pubmedqa.jsonl", "eval_mmlu_medical.jsonl", "data_report.json")


def find_code_dir(root: Path, required: set[str] = REQUIRED) -> Path:
    """Locate the uploaded modules wherever Kaggle mounted them.

    The mount path is not stable: a dataset declared as gb1105/medical-ft-code
    turned up under /kaggle/input/datasets/... rather than at
    /kaggle/input/medical-ft-code. Two runs died on that assumption, so this
    searches for the directory that actually holds the modules instead.
    """
    if not root.exists():
        raise SystemExit(
            "\nSTOP. /kaggle/input does not exist -- no inputs are attached.\n"
            "FIX: sidebar -> + Add Input -> Datasets -> medical-ft-code.")
    candidates = []
    for path in root.rglob("*.py"):
        stems = {p.stem for p in path.parent.glob("*.py")}
        if required <= stems:
            candidates.append(path.parent)
    if not candidates:
        found = sorted(str(p.relative_to(root)) for p in root.rglob("*.py"))[:20]
        tree = sorted(str(p.relative_to(root)) for p in root.rglob("*"))[:30]
        raise SystemExit(
            f"\nSTOP. No directory under {root} contains all of {sorted(required)}.\n"
            f".py files found: {found or 'none'}\n"
            f"First entries under /kaggle/input: {tree}\n"
            "FIX: re-run scripts/push_kaggle.sh, then confirm the "
            "medical-ft-code dataset is attached in the sidebar.")
    return sorted(set(candidates))[0]


def find_adapter_dir(root: Path) -> Path:
    """Locate the uploaded LoRA adapter by content, the same way.

    A directory qualifies when adapter_config.json and adapter_model.safetensors
    sit side by side. A final adapter wins over any checkpoint-* directory, so a
    stray checkpoint cannot be scored in place of the finished run.
    """
    if not root.exists():
        raise SystemExit(
            "\nSTOP. /kaggle/input does not exist -- no inputs are attached.\n"
            "FIX: sidebar -> + Add Input -> Datasets -> medical-ft-adapter.")
    found = sorted({p.parent for p in root.rglob("adapter_config.json")
                    if (p.parent / "adapter_model.safetensors").exists()})
    final = [d for d in found if not d.name.startswith("checkpoint-")]
    if final or found:
        return (final or found)[0]
    tree = sorted(str(p.relative_to(root)) for p in root.rglob("*"))[:30]
    raise SystemExit(
        f"\nSTOP. No LoRA adapter under {root}: no adapter_config.json with "
        "adapter_model.safetensors beside it.\n"
        f"First entries under /kaggle/input: {tree}\n"
        "FIX: run scripts/push_adapter.sh, then attach medical-ft-adapter in "
        "the sidebar.")


def code_fingerprint(directory: Path) -> str:
    """A short hash of every .py file's name and bytes, in name order.

    Each notebook is built for one exact set of modules. If Kaggle mounts an
    older version of the code dataset -- a new version still processing, or an
    old one attached by hand -- the modules carry the right names and the wrong
    code, and nothing else would notice.
    """
    import hashlib

    digest = hashlib.sha256()
    for path in sorted(directory.glob("*.py")):
        digest.update(path.name.encode() + b"\0" + path.read_bytes() + b"\0")
    return digest.hexdigest()[:16]


def find_data_dir(root: Path) -> Path:
    """Locate the uploaded run 3 data by content, like find_code_dir."""
    if not root.exists():
        raise SystemExit(
            "\nSTOP. /kaggle/input does not exist -- no inputs are attached.\n"
            "FIX: sidebar -> + Add Input -> Datasets -> medical-ft-data.")
    found = sorted({p.parent for p in root.rglob("train.jsonl")
                    if all((p.parent / name).exists() for name in DATA_FILES)})
    if found:
        return found[0]
    tree = sorted(str(p.relative_to(root)) for p in root.rglob("*"))[:30]
    raise SystemExit(
        f"\nSTOP. No directory under {root} holds all of {list(DATA_FILES)}.\n"
        f"First entries under /kaggle/input: {tree}\n"
        "FIX: run scripts/push_data.sh, then attach medical-ft-data in the sidebar.")


def data_fingerprint(directory: Path) -> str:
    """The same idea as code_fingerprint, for the frozen data files."""
    import hashlib

    digest = hashlib.sha256()
    for name in DATA_FILES:
        digest.update(name.encode() + b"\0" + (directory / name).read_bytes() + b"\0")
    return digest.hexdigest()[:16]

SRC = find_code_dir(INPUT, RUN2_REQUIRED)
EXPECTED_FINGERPRINT = "9c7078cfa7b59bc4"
if code_fingerprint(SRC) != EXPECTED_FINGERPRINT:
    raise SystemExit(
        f"\nSTOP. The attached code ({code_fingerprint(SRC)}) is not the code this "
        f"notebook was built for ({EXPECTED_FINGERPRINT}).\n"
        "Kaggle may still be processing a new version of medical-ft-code, or an "
        "older version is attached.\n"
        "FIX: wait a minute and re-run; if it persists, run scripts/push_kaggle.sh again.")
DATA_SRC = find_data_dir(INPUT)
EXPECTED_DATA_FINGERPRINT = "ee31015e2db1fe73"
if data_fingerprint(DATA_SRC) != EXPECTED_DATA_FINGERPRINT:
    raise SystemExit(
        f"\nSTOP. The attached data ({data_fingerprint(DATA_SRC)}) is not the data "
        f"this notebook was built for ({EXPECTED_DATA_FINGERPRINT}).\n"
        "FIX: wait a minute and re-run; if it persists, run scripts/push_data.sh again.")
print("code:", SRC)
print("data:", DATA_SRC)

for src_file in sorted(SRC.glob("*.py")):
    shutil.copy(src_file, PKG / src_file.name)
(PKG / "__init__.py").touch()
for name in DATA_FILES:
    shutil.copy(DATA_SRC / name, DATA / name)

os.chdir(WORK)
sys.path.insert(0, str(WORK))
Path("outputs").mkdir(exist_ok=True)

REPORT = json.loads((DATA / "data_report.json").read_text())
MAX_SEQ = REPORT["max_seq"]
print(f"train {REPORT['train_size']:,} examples | max_seq {MAX_SEQ} | reasoning "
      f"{REPORT['reasoning_fraction']:.0%} | multiple choice {REPORT['mcq_fraction']:.0%}")
for label, stats in REPORT["sources"].items():
    print(f"  {label:<24} kept {stats['kept']:>6,} of target {stats['target']:>6,}")

# No shell: every command here is built from constants, so it splits into a
# plain list. "python" becomes this kernel's own interpreter -- the one the
# install cell installed Unsloth into -- rather than whatever PATH finds.
def argv(cmd: str) -> list[str]:
    args = shlex.split(cmd)
    return [sys.executable] + args[1:] if args[0] == "python" else args

# A failing command returns non-zero without raising in Jupyter.
def step(cmd: str, env: dict | None = None):
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.run(argv(cmd), env={**os.environ, **(env or {})})
    if p.returncode != 0:
        raise SystemExit(f"\nStep failed (exit {p.returncode}):\n  {cmd}")
    print("\nok\n", flush=True)

## 4. Train

The thinking format on GPU 0, with a time guard that saves the adapter instead of overrunning the session.

In [ ]:
# --- 4. Train on GPU 0 -------------------------------------------------------
# Leave the evaluation about 4.25 hours, and never train for more than 5.75.
STOP_AFTER = int(min(5.75 * 3600, DEADLINE - time.time() - 4.25 * 3600))
if STOP_AFTER < 2 * 3600:
    raise SystemExit(f"\nSTOP. Only {STOP_AFTER / 3600:.1f}h are left for training.\n"
                     "FIX: setup was unusually slow; Run All again.")
print(f"training stops by itself after {STOP_AFTER / 3600:.2f}h at the latest")

step(f"python -m training.train --format v2 --data data/v2 --out outputs/run3 "
     f"--max-seq {MAX_SEQ} --batch-size 2 --grad-accum 8 --rank 64 --lr 1e-4 "
     f"--epochs 1 --save-steps 200 --group-by-length --probe-steps 20 "
     f"--no-probe-abort --stop-after-seconds {STOP_AFTER}",
     env={"CUDA_VISIBLE_DEVICES": "0"})

In [ ]:
# --- 5. Save the adapter and the training record to the Output panel -------
import json, shutil

OUT = Path("/kaggle/working")
shutil.copytree("outputs/run3", OUT / "run3-adapter", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("checkpoint-*"))
for name in ("train_stats.json", "loss_curve.json"):
    shutil.copy(Path("outputs/run3") / name, OUT / name)
shutil.copy(DATA / "data_report.json", OUT / "data_report.json")
print("########## Training ##########")
for key, value in json.loads((OUT / "train_stats.json").read_text()).items():
    print(f"  {key}: {value}")

## 6. Evaluate

A smoke run of every stage first, then the full run: letter choice on all 7,545 questions, then reasoning, until the deadline.

In [ ]:
# --- 6. Evaluation smoke run: every stage, four questions, both models -----
def worker(role: str, gpu: int, out_dir: str, extra: str):
    cmd = (f"python -m training.eval_worker --role {role} --eval-dir data/v2 "
           f"--out-dir {out_dir} --device cuda "
           + ("--adapter outputs/run3 " if role == "tuned" else "") + extra)
    log = open(f"{out_dir}-{role}.log", "w")
    proc = subprocess.Popen(argv(cmd), stdout=log, stderr=subprocess.STDOUT,
                            env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu),
                                 # hours of variable-length batches fragment memory
                                 "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
    return proc, log

def run_pair(out_dir: str, extra_tuned: str, extra_base: str, poll: int) -> dict:
    """Both models: in parallel on two GPUs, one after the other on one."""
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    if N_GPUS > 1:
        procs = {"tuned": worker("tuned", 0, out_dir, extra_tuned),
                 "base": worker("base", 1, out_dir, extra_base)}
        while any(p.poll() is None for p, _ in procs.values()):
            time.sleep(poll)
            written = {role: sum(1 for f in Path(out_dir, role).glob("*.jsonl")
                                 for _ in open(f)) for role in procs}
            print(time.strftime("%H:%M"), "answers written:", written, flush=True)
        codes = {role: p.returncode for role, (p, _) in procs.items()}
    else:
        codes = {}
        for role, extra in (("tuned", extra_tuned), ("base", extra_base)):
            proc, _ = worker(role, 0, out_dir, extra)
            codes[role] = proc.wait()
    for role in ("tuned", "base"):
        print(f"--- {role} log tail ---")
        print("".join(open(f"{out_dir}-{role}.log").readlines()[-15:]))
    return codes

SMOKE = "--limit 4 --think-budget 128 --batch-size 4"
codes = run_pair("outputs/eval_smoke", SMOKE, SMOKE, poll=15)
if any(codes.values()):
    raise SystemExit(f"\nSTOP. The evaluation smoke run failed: {codes}. The adapter "
                     "is already in the Output panel; the log tails above say why.")
step("python -m training.eval_report --eval-dir outputs/eval_smoke --bench-dir data/v2 "
     "--out outputs/eval_smoke/report.json")

In [ ]:
# --- 7. Full evaluation, until the deadline ---------------------------------
SETTINGS = "--think-budget 1536 --batch-size 16 --seed 1234"
if N_GPUS > 1:
    codes = run_pair("outputs/eval", f"--deadline {DEADLINE:.0f} {SETTINGS}",
                     f"--deadline {DEADLINE:.0f} {SETTINGS}", poll=600)
else:
    half = time.time() + (DEADLINE - time.time()) / 2
    codes = run_pair("outputs/eval", f"--deadline {half:.0f} {SETTINGS}",
                     f"--deadline {DEADLINE:.0f} {SETTINGS}", poll=600)
print("worker exit codes:", codes)
step("python -m training.eval_report --eval-dir outputs/eval --bench-dir data/v2 "
     "--out /kaggle/working/run3_eval.json")
shutil.copytree("outputs/eval", "/kaggle/working/run3_eval_predictions",
                dirs_exist_ok=True)
print("Saved run3_eval.json and every prediction to the Output panel.")

## Done

In the **Output** panel: `run3-adapter/`, `train_stats.json`, `loss_curve.json`,
`data_report.json`, `run3_eval.json` (every stage and the pooled results, with
McNemar p and a 95% interval for each change) and `run3_eval_predictions/`
(every question's answer from both models, with the reasoning text).